# 01 — Crypto trend lab (spot + 4h TEMA)

Research only. QMIE remains signal-only. **Do not retune live `W_*`.** Frozen TEMA is 9/90/199.

## Protocol (this is the test, not a fit story)

The request “2018–2023 OOS, 2023→now IS” trains on the **future**. That is a leakage diagnostic, not a valid holdout. This lab fits **IS 2019-09 → 2022-12** and never tunes on **OOS 2023 → today**. Vision USDT-M starts ~2019-09, not 2018. `WARMUP_BARS = 220`. Positions are `signal.shift(1)`.

## Hypotheses in this notebook

| Id | Claim |
|---|---|
| H1 | Spot 1D EMA+Donchian+ADX beats buy-and-hold on OOS Sharpe **or** tighter DD |
| H2 | 10× isolated TEMA raises expectancy vs 1× but worsens max DD / liquidations |
| H3 | Optuna-best TEMA periods fail DF / OOS vs frozen 9/90/199 — **do not promote** |
| H4 | Boruta-confirmed confluence (KAMA / MACD / z-score / ALMA) improves OOS vs raw breakout |

Promote-to-live requires IS Sharpe **and** DF neighborhood **and** OOS holdout. None of these cells write scanner weights.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif (ROOT / "research").exists():
    pass
elif (ROOT / "python" / "research").exists():
    ROOT = ROOT / "python"
sys.path.insert(0, str(ROOT))
print("python root", ROOT)


In [ ]:
from research.trend_lab.protocol import SPLIT, WARMUP_BARS
from research.trend_lab.data import CORE, coverage_table, load_symbol
from research.trend_lab.evaluate import eval_spot, eval_tema, reverse_split_diagnostic
from research.trend_lab.features import feature_frame
from research.trend_lab.optimize import boruta_select, df_neighborhood_score, grid_spot, grid_tema, optimize_spot, optimize_tema, trend_label
from research.trend_lab.protocol import inner_validation_start, split_frame
from research.trend_lab.spot_system import SpotParams, spot_signal
from research.trend_lab.tema_system import TemaParams
from research.trend_lab.plots import df_scatter, equity_overlay, param_heatmap, price_signals, rolling_sharpe_fig, underwater
from scanner.indicators import ema
from research.trend_lab.features import alma, kama
import pandas as pd

print(SPLIT)
print("warmup", WARMUP_BARS)
print(SPLIT.requested_note)


In [ ]:
cov = coverage_table(CORE[:5], ("1d", "4h"))
display(cov)


In [ ]:
btc_1d, src_1d = load_symbol("BTCUSDT", "1d")
btc_4h, src_4h = load_symbol("BTCUSDT", "4h")
print("1d", src_1d, len(btc_1d), btc_1d.index[0] if len(btc_1d) else None, "→", btc_1d.index[-1] if len(btc_1d) else None)
print("4h", src_4h, len(btc_4h), btc_4h.index[0] if len(btc_4h) else None, "→", btc_4h.index[-1] if len(btc_4h) else None)
parts = split_frame(btc_1d)
print("IS bars", len(parts["is"]), "OOS bars", len(parts["oos"]))


## Boruta on IS only

Labels are next-10-bar sign of return. The last 10 IS bars are dropped so the label never uses OOS closes. Shadows are permuted copies of the same features. Confirmed = beat max shadow in ≥95% of iterations.


In [ ]:
is_1d = parts["is"]
feats = feature_frame(is_1d).iloc[WARMUP_BARS:]
y = trend_label(is_1d["close"], horizon=10).reindex(feats.index)
boruta = boruta_select(feats.iloc[:-10], y.iloc[:-10], n_iter=8, n_estimators=80)
display(boruta)
confirmed = boruta.loc[boruta.decision.eq("confirmed"), "feature"].tolist()
print("confirmed", confirmed)


## Approach 1 — spot (leverage 1)

Radar analog: fast EMA > slow EMA, **prior-window** Donchian breakout, ADX ≥ min and +DI > −DI, RSI cap, hold while above prior box. Grid + Optuna (or random search if Optuna is missing) fit **IS only**. DF neighborhood is scored on the last 20% of IS, never OOS.


In [ ]:
baseline = SpotParams()
grid = grid_spot(is_1d)
display(grid.head(8))
opt = optimize_spot(is_1d, n_trials=16)
print("engine", opt.get("engine"), opt["params"])
base_ev = eval_spot(btc_1d, baseline)
opt_ev = eval_spot(btc_1d, opt["params"])
conf_p = SpotParams(
    use_kama="kama_cross" in confirmed or "kama_er" in confirmed,
    use_macd="macd_hist" in confirmed,
    use_zscore="zscore_20" in confirmed,
    use_alma="alma_slope" in confirmed,
)
conf_ev = eval_spot(btc_1d, conf_p)
summary = pd.DataFrame({
    "baseline_IS": base_ev["is"], "baseline_OOS": base_ev["oos"],
    "optuna_OOS": opt_ev["oos"], "confluence_OOS": conf_ev["oos"],
    "BH_OOS": base_ev["bh_oos"],
}).T
display(summary.round(3))


In [ ]:
inner = inner_validation_start(is_1d.index)
center = {
    "ema_fast": float(opt["params"].ema_fast),
    "ema_slow": float(opt["params"].ema_slow),
    "donchian": float(opt["params"].donchian),
    "min_adx": float(opt["params"].min_adx),
}
steps = {k: [v-d, v, v+d] for (k, v), d in zip(center.items(), (2, 10, 5, 2))}

def run_fn(ohlcv, p):
    fr = spot_signal(ohlcv, SpotParams(ema_fast=int(p["ema_fast"]), ema_slow=int(p["ema_slow"]), donchian=int(p["donchian"]), min_adx=float(p["min_adx"])))
    return fr[["net", "equity"]]

dfn = df_neighborhood_score(is_ohlcv=is_1d, inner_val_start=inner, center=center, neighbor_steps=steps, run_fn=run_fn, min_is_sharpe=0.5)
print(dfn["status"], "val_sharpe_std", dfn.get("val_sharpe_std"), "n_stable", dfn.get("n_stable"))
if dfn.get("table") is not None and len(dfn["table"]):
    df_scatter(dfn["table"], "Spot DF neighborhood (inner IS)").show()
param_heatmap(grid, "ema_slow", "ema_fast", "sharpe", "Spot grid IS Sharpe").show()


In [ ]:
equity_overlay({
    "spot baseline": base_ev["oos_frame"]["equity"],
    "spot Optuna": opt_ev["oos_frame"]["equity"],
    "confluence": conf_ev["oos_frame"]["equity"],
    "buy&hold": (1 + parts["oos"]["close"].pct_change().fillna(0)).cumprod(),
}, "OOS growth of $1 — BTC spot").show()
rolling_sharpe_fig({"spot": base_ev["oos_frame"]["net"], "optuna": opt_ev["oos_frame"]["net"]}, 90, "OOS 90d rolling Sharpe").show()
underwater(base_ev["oos_frame"]["equity"], "Spot baseline OOS DD").show()

oos = parts["oos"]
ks, _ = kama(oos["close"], 10)
overlays = {"EMA9": ema(oos["close"], 9), "EMA199": ema(oos["close"], 199), "KAMA10": ks, "ALMA9": alma(oos["close"], 9)}
entries = base_ev["oos_frame"].index[base_ev["oos_frame"]["signal"].diff().fillna(0) > 0]
exits = base_ev["oos_frame"].index[base_ev["oos_frame"]["signal"].diff().fillna(0) < 0]
price_signals(oos, signal=base_ev["oos_frame"]["held"], entries=entries, exits=exits, overlays=overlays, title="BTC 1D OOS — spot vs KAMA/ALMA/EMA").show()


## Approach 2 — 4h TEMA, isolated 10×

Same-bar SL and TP → SL. Loss capped at stake. Frozen 9/90/199 is always reported. Optuna search is **research**; `do_not_promote=True`.


In [ ]:
t10 = eval_tema(btc_4h, TemaParams(leverage=10.0))
t1 = eval_tema(btc_4h, TemaParams(leverage=1.0))
gt = grid_tema(split_frame(btc_4h)["is"], leverage=10.0)
display(gt)
ot = optimize_tema(split_frame(btc_4h)["is"], n_trials=12, leverage=10.0)
ot_ev = eval_tema(btc_4h, ot["params"])
print("do_not_promote", ot["do_not_promote"], "engine", ot.get("engine"))
tema_tbl = pd.DataFrame({
    "frozen_10x_IS": t10["is"], "frozen_10x_OOS": t10["oos"],
    "frozen_1x_OOS": t1["oos"], "optuna_10x_OOS": ot_ev["oos"],
    "frozen_10x_IS_opt_report": ot["frozen_9_90_199_is"], "optuna_10x_IS": ot["is_kpis"],
}).T
display(tema_tbl.round(3))
oos4 = split_frame(btc_4h)["oos"]
price_signals(oos4, entries=pd.DatetimeIndex(t10["oos_trades"]["entry_time"]) if len(t10["oos_trades"]) else None,
              exits=pd.DatetimeIndex(t10["oos_trades"]["exit_time"]) if len(t10["oos_trades"]) else None,
              title="BTC 4h OOS — frozen TEMA 9/90/199 10x isolated", max_bars=800).show()


## Leakage diagnostic (do not select from this)

Train on 2023→now, test on 2019–2022. If this looks better than the chronological OOS, that is **overfit theatre**, not edge.


In [ ]:
leak = reverse_split_diagnostic(btc_1d, baseline)
print(leak["note"])
display(pd.DataFrame({"fit_on_future": leak["fit_on_future"], "test_on_past": leak["test_on_past"]}).T.round(3))
